# HTR with Claude

Requires an API key from platform.claude.ai

In [17]:
import anthropic
from datetime import datetime
from difflib import SequenceMatcher
from dotenv import load_dotenv
import json
import os
from pathlib import Path
import regex

from IPython.display import clear_output

In [2]:
def squeal(text=None):
    clear_output(wait=True)
    if not text is None: print(text)

In [3]:
base_directory = "../memories_crawl/scans/bhic/Eindhoven/deel_84"

## 1. Find act-initial text block with Claude

Processing a single image costs about 0.5 cents

In [ ]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
prompt = """Dear Claude, here is an image displaying two pages with Dutch text 
related to an inheritance. I am interested in the information in the text block 
on the top right of the right page. Could check if that part contains a text like: 
"Memorie van aangifte der nalatenschap van"? If that is the case, can you give me 
the information which follows next? This is 1. the name of the deceased, 2. the 
place of death, and 3. the date of death. Both place and date could be missing. 
If you see a big number next to the text block, that is the act number, which is 
interesting as well. Please return this all information in well-formatted JSON 
format with the keys "act", "name", "place" and "date" and without any comments. 
If the top right of the right page contains a different text or no text at all, 
please return an empty JSON structure."""

In [ ]:
def clear_claude_storage():
    files = client.beta.files.list()
    for file in files:
        client.beta.files.delete(file.id)

In [ ]:
def send_claude_prompt(prompt, upload_file_name):
    try:
        upload_response = client.beta.files.upload(file=Path(upload_file_name))
        message = client.beta.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            messages=[
                {"role": "user", 
                 "content": [
                    {"type": "image",
                     "source": {"type": "file",
                                "file_id": upload_response.id
                               }
                    },
                    {"type": "text", "text": prompt}
                  ]}],
            betas=["files-api-2025-04-14"]
        )
    finally:
        clear_claude_storage()  
    return message.content[0].text

In [ ]:
def add_comment(person_dict, comment_prefix, comment_suffix):
    if comment_prefix:
        if comment_suffix:
            person_dict["comment"] = " ".join([comment_prefix, comment_suffix])
        else:
            person_dict["comment"] = comment_prefix
    elif comment_suffix:
        person_dict["comment"] = comment_suffix

In [ ]:
def add_page_number(person_dict, page_number, sample_file_name):
    sample_file_name_parts = regex.split(r"[_.]", sample_file_name)
    sample_file_name_parts[-2] = str(page_number).zfill(len(sample_file_name_parts[-2]))
    sample_file_name_parts[-2] += "." + sample_file_name_parts[-1]
    sample_file_name_parts.pop()
    person_dict["scan_file"] = "_".join(sample_file_name_parts)
    person_dict["page_number"] = page_number

In [ ]:
def str2dict(string, page_number, sample_file_name):
    groups = regex.search(r"^(.*)```json(.*)```(.*)$", string.strip(), flags=regex.DOTALL)
    person_dict = json.loads(groups.group(2))
    add_comment(person_dict, groups.group(1).strip(), groups.group(1).strip())
    add_page_number(person_dict, page_number, sample_file_name)
    return person_dict

In [ ]:
def save_json(results_json):
    today = datetime.strftime(datetime.now(), "%Y%m%d")
    with open(f"output_{today}.json", "w") as f:
        json.dump(results_json, f)

In [ ]:
def claude2json(results):
    results_json = []
    for page_number, result in results.items():
        results_json.append(str2dict(result, page_number, sample_file_name))
    return results_json

Process scans with Claude. If the block crashes, just run it again to continue

In [ ]:
page_number = 1 if "page_number" not in globals()
starting_pages = [x for x in range(page_number, 469)]
sample_file_name = ""
for file_name in sorted(os.listdir(base_directory)):
    try:
        page_number = int(regex.sub("^0+", "", file_name.split("_")[-1].split('.')[0]))
    except ValueError:
        continue
    if page_number in starting_pages:
        results[page_number] = send_claude_prompt(prompt, os.path.join(base_directory, file_name))
        sample_file_name = file_name
        squeal(page_number)

In [ ]:
results_json = claude2json(results)
save_json(results_json)

## 2. Link HTR information to metadata

In [5]:
def read_json(file_name):
    with open(file_name, "r") as infile:
        return json.load(infile)

In [18]:
def string_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

In [6]:
claude_data = read_json("output_20260521.json")
meta_data = read_json(os.path.join(base_directory, "deeds.json"))

In [50]:
nbr_of_exact_matches = 0
nbr_of_close_matches = 0
unique_claude_names = set([scan["name"] for scan in claude_data if "name" in scan and scan["name"]])
for target_name in unique_claude_names:
    max_similarity = 0
    closest_match = ""
    found = False
    for act in meta_data:
        # if naam_volledig in person and person.naam_volledig == target_name:
        if "personen" in act:
            for person in act["personen"]:
                if "naam_volledig" in person:
                    if person["naam_volledig"] == target_name:
                        print(f"found {target_name}")
                        nbr_of_exact_matches += 1
                        found = True
                    else:
                        similarity = string_similarity(person["naam_volledig"], target_name)
                        if similarity > max_similarity:
                            max_similarity = similarity
                            closest_match = person["naam_volledig"]
    if not found:
        print(f"missed {target_name}; closest match({max_similarity:0.1f}): {closest_match}")
        if max_similarity >= 0.9:
            nbr_of_close_matches += 1
print(f"number of unique names: {len(unique_claude_names)}")
print(f"exact matches: {nbr_of_exact_matches} ({100*nbr_of_exact_matches/len(unique_claude_names):0.0f}%)")
print(f"close matches: {nbr_of_close_matches} ({100*nbr_of_close_matches/len(unique_claude_names):0.0f}%)")

found Johannes van Mierlo
missed Knubers; closest match(0.5): Martinus Beks
missed Freen (Jansdochter/Johannesdochter); closest match(0.6): Peter Johannes Custers
missed Joannes Cornelis Vankloof; closest match(0.9): Joannis Cornelis van Hoof
missed Adrianus Green; closest match(0.9): Adrianus Freen
found Petronella Smolders
found Francis Sengers
found Johannes Aarts
missed Johannes Lathouvers; closest match(0.9): Johannes Lathouwers
missed van den Boomen, Johannes; closest match(0.6): Johannes van den Boomen
missed Moynen Johannes Antonius; closest match(0.7): Johannes Antonius Moonen
missed Van Bovy; closest match(0.5): Anna Tiebos
found Antonius Schellens
missed Maria Catharina van den Broeck; closest match(1.0): Maria Catharina van den Broek
missed Johanna van den Veldevoort; closest match(0.8): Johanna van der Velden
missed Joannes Cornelis den Hoof; closest match(0.9): Joannis Cornelis van Hoof
missed Christina de Krom weduwe Johannes Sanders; closest match(0.6): Christina de Kro